# Check Data Completeness

This notebook investigates the completeness of "exp-static-flexible-anon-2025-05-29" and justifies the implementation of tidy_data.py.

Main findings: the logged have missing weeks. There is no null values for the proficient column, which logs the number of skills learned per week.

In [8]:
import pandas as pd
import numpy as np

def check_data_completeness():
    """
    Check if every student has complete rows with weeks from [-2,-1,...,8] 
    and corresponding 'proficient' values.
    """
    print("Analyzing data completeness for student time series...")
    print("=" * 60)
    
    # Load the data
    try:
        # Try data_tidied.csv first as it looks more structured for time series
        df = pd.read_csv('~/cmu/goalsetting-recommendation-algorithm/time-series-predictor/data/data_tidied.csv')
        print(f"Loaded data_tidied.csv with {len(df)} rows")
        print(f"Columns: {list(df.columns)}")
        
    except FileNotFoundError:
        print("data_tidied.csv not found, trying alternative file...")
        try:
            df = pd.read_csv('exp-static-flexible-anon-2025-05-29.csv')
            print(f"Loaded exp-static-flexible-anon-2025-05-29.csv with {len(df)} rows")
            print(f"Columns: {list(df.columns)}")
        except FileNotFoundError:
            print("Neither data file found!")
            return
    
    print("\nFirst few rows:")
    print(df.head(10))
    
    # Define expected week range
    expected_weeks = list(range(-2, 9))  # [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
    print(f"\nExpected weeks: {expected_weeks}")
    print(f"Total expected weeks per student: {len(expected_weeks)}")
    
    # Check unique weeks in data
    unique_weeks = sorted(df['week'].unique())
    print(f"Actual weeks in data: {unique_weeks}")
    
    # Check unique students
    unique_students = df['name'].unique()
    print(f"Number of unique students: {len(unique_students)}")
    
    # Check for missing weeks across all students
    missing_weeks = set(expected_weeks) - set(unique_weeks)
    if missing_weeks:
        print(f"⚠️  Missing weeks in entire dataset: {sorted(missing_weeks)}")
    else:
        print("✅ All expected weeks present in dataset")
    
    print("\n" + "=" * 60)
    print("STUDENT-LEVEL COMPLETENESS ANALYSIS")
    print("=" * 60)
    
    # Analyze each student's data completeness
    complete_students = []
    incomplete_students = []

    null_exists = False
    
    for student in unique_students:
        student_data = df[df['name'] == student]
        student_weeks = set(student_data['week'].unique())
        missing_weeks_student = set(expected_weeks) - student_weeks
        
        # Check for null proficient values
        null_proficient = student_data['proficient'].isnull().sum()
        if null_proficient > 0:
            null_exists = True
        
        if len(missing_weeks_student) == 0 and null_proficient == 0:
            complete_students.append(student)
        else:
            incomplete_students.append({
                'student': student,
                'missing_weeks': sorted(missing_weeks_student),
                'null_proficient': null_proficient,
                'total_rows': len(student_data)
            })
    
    # Summary results
    print(f"✅ Students with COMPLETE data: {len(complete_students)}")
    print(f"❌ Students with INCOMPLETE data: {len(incomplete_students)}")
    print(f"📊 Completeness rate: {len(complete_students)/len(unique_students)*100:.1f}%")

    print(f"Null exists: {null_exists}")
    
    # Show details for incomplete students
    if incomplete_students:
        print(f"\nDETAILS FOR INCOMPLETE STUDENTS:")
        print("-" * 50)
        for i, student_info in enumerate(incomplete_students[:10]):  # Show first 10
            print(f"{i+1}. Student: {student_info['student'][:]}")
            if student_info['missing_weeks']:
                print(f"   Missing weeks: {student_info['missing_weeks']}")
            if student_info['null_proficient'] > 0:
                print(f"   Null proficient values: {student_info['null_proficient']}")
            print(f"   Total rows: {student_info['total_rows']}")
            print()
        
        if len(incomplete_students) > 10:
            print(f"... and {len(incomplete_students) - 10} more students with incomplete data")
    
    # Show sample of complete students
    if complete_students:
        print(f"\nSAMPLE OF COMPLETE STUDENTS:")
        print("-" * 30)
        for i, student in enumerate(complete_students[:5]):
            student_data = df[df['name'] == student]
            print(f"{i+1}. Student: {student[:]} ({len(student_data)} rows)")
            weeks_with_proficient = student_data[['week', 'proficient']].sort_values('week')
            print(f"   Weeks: {sorted(weeks_with_proficient['week'].tolist())}")
            print(f"   Proficient values: {weeks_with_proficient['proficient'].tolist()}")
            print()
    
    # Answer the specific question
    print("\n" + "=" * 60)
    print("ANSWER TO YOUR QUESTION")
    print("=" * 60)
    
    if len(incomplete_students) == 0:
        print("✅ YES - Every student has complete rows with weeks from [-2,-1,...,8]")
        print("   and corresponding 'proficient' values.")
    else:
        print("❌ NO - Not every student has complete data.")
        print(f"   {len(incomplete_students)} out of {len(unique_students)} students")
        print("   are missing data for some weeks but none but none have null proficient values.")
    
    return {
        'total_students': len(unique_students),
        'complete_students': len(complete_students),
        'incomplete_students': len(incomplete_students),
        'completeness_rate': len(complete_students)/len(unique_students)*100,
        'expected_weeks': expected_weeks,
        'actual_weeks': unique_weeks
    }


results = check_data_completeness() 

Analyzing data completeness for student time series...
Loaded data_tidied.csv with 1893 rows
Columns: ['name', 'week', 'proficient']

First few rows:
                               name  week  proficient
0  005925abe62fc0d85a44032f85cd2465    -1         0.0
1  005925abe62fc0d85a44032f85cd2465     0         3.0
2  005925abe62fc0d85a44032f85cd2465     1         0.0
3  005925abe62fc0d85a44032f85cd2465     2         0.0
4  005925abe62fc0d85a44032f85cd2465     3         2.0
5  005925abe62fc0d85a44032f85cd2465     4         1.0
6  005925abe62fc0d85a44032f85cd2465     5         3.0
7  005925abe62fc0d85a44032f85cd2465     6         4.0
8  005925abe62fc0d85a44032f85cd2465     7         2.0
9  005925abe62fc0d85a44032f85cd2465     8         1.0

Expected weeks: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
Total expected weeks per student: 11
Actual weeks in data: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
Number of unique students: 184
✅ All expected weeks present in dataset

STUDENT-LEVEL COMPLETENESS ANALYSIS
